# processing에 멈춘 OCR 캐시 문서 상태 강제 수정

재배포/워커 크래시 때문에 `status`가 `processing`에서 멈춘 채로 안 넘어가는 문서를, id를 알고 있을 때 `failed`로 되돌리는 노트북입니다.

**주의: `ai_search_test.ipynb`와 달리 여기서는 `-test` 인덱스가 아니라 실제 서비스 인덱스(`app/services/search_index_service.py`가 쓰는 그 인덱스)를 직접 수정합니다.** id를 잘못 넣으면 다른 문서를 건드릴 수 있으니 셀 순서대로, 출력 확인해가며 실행하세요.

## 사전 준비
1. `pip install azure-search-documents python-dotenv`
2. 프로젝트 루트 `.env`에 `AZURE_SEARCH_ENDPOINT`, `AZURE_SEARCH_API_KEY`(admin key), `AZURE_SEARCH_INDEX_NAME`이 채워져 있어야 합니다.

## 수정 후
`failed`로 바뀌면 `ocr_cache_service.claim()`(`app/services/ocr_cache_service.py:59-69`) 로직상 다음에 같은 문서(같은 content_hash)가 다시 요청으로 들어올 때 `_reclaim()`을 타고 재처리됩니다.

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents import SearchClient
from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
API_KEY = os.environ["AZURE_SEARCH_API_KEY"]
INDEX_NAME = os.environ.get("AZURE_SEARCH_INDEX_NAME", "ocr-documents")

search_client = SearchClient(endpoint=ENDPOINT, index_name=INDEX_NAME, credential=AzureKeyCredential(API_KEY))

print("ENDPOINT:", ENDPOINT)
print("INDEX_NAME:", INDEX_NAME, "(실제 서비스 인덱스, -test 아님)")

## 1. 대상 문서 조회

`STUCK_ID`에 멈춰있는 문서의 `id`(= `{content_hash}_{model_id}_{output_format}`)를 채우고 실행하세요.

In [ ]:
STUCK_ID = "<여기에 멈춘 문서의 id를 채우세요>"

doc = search_client.get_document(key=STUCK_ID)
print("현재 문서 상태:")
for k, v in doc.items():
    print(f"  {k}: {v}")

## 2. 안전 체크

`status`가 실제로 `processing`일 때만 다음 셀로 진행하세요. 이미 `completed`/`failed`인 문서를 실수로 덮어쓰는 걸 막기 위한 확인용입니다.

In [ ]:
assert doc["status"] == "processing", f"status가 processing이 아닙니다 (현재: {doc['status']}). 정말 바꿀 게 맞는지 다시 확인하세요."
print("OK - status가 processing인 것 확인. 다음 셀에서 failed로 바꿉니다.")

## 3. failed로 갱신

`merge_or_upload_documents`는 넘긴 필드만 덮어쓰고 나머지(원래 파일명, 크기 등)는 그대로 유지합니다.

In [ ]:
REASON = "서버 재배포로 인해 처리 중 워커 프로세스가 종료되어 중단됨"

search_client.merge_or_upload_documents(
    [
        {
            "id": STUCK_ID,
            "status": "failed",
            "error_message": REASON,
            "updated_at": datetime.now(timezone.utc).isoformat(),
        }
    ]
)
print(f"'{STUCK_ID}' -> failed로 갱신 완료")

## 4. 재조회로 확인

In [ ]:
updated_doc = search_client.get_document(key=STUCK_ID)
print("갱신 후 문서 상태:")
for k, v in updated_doc.items():
    print(f"  {k}: {v}")